# Folding proteins on your own GPU

**Biomolecular structure prediction with ESMFold2 — 25 August 2026**

You have just seen these systems folded through the Biohub endpoint. We are going
to fold the same ones on a local GPU — a protein, a protein bound to RNA and DNA,
a drug molecule attached to its receptor, and a two-chain complex with alignments.

The point of the next ten minutes is not the biology. It is that **all four look
almost identical in code.** Once you can write one, you can write all of them.


## Setup


In [ ]:
# Colab: uncomment.
# !pip install -q esm

from esm.models.esmfold2 import ESMFold2InputBuilder, EsmFold2Model

model = EsmFold2Model.from_pretrained("biohub/ESMFold2-Fast", device="cuda").eval()
model.set_kernel_backend("fused")  # optional: faster on anything over ~300 residues

builder = ESMFold2InputBuilder()

The viewing helpers, using the same pLDDT bands and colours as [esmfold2.ipynb](esmfold2.ipynb). No ESMFold2 in here — run it and move on.


In [ ]:
import numpy as np
import py3Dmol
from IPython.display import HTML, display

# Same pLDDT bands and colours as cookbook/tutorials/esmfold2.ipynb.
PLDDT_LEGEND = (
    '<span style="color:#FF7D45;">&#9632;</span> &lt;50 &nbsp;'
    '<span style="color:#FFDB13;">&#9632;</span> 50–70 &nbsp;'
    '<span style="color:#65CBF3;">&#9632;</span> 70–90 &nbsp;'
    '<span style="color:#0053D6;">&#9632;</span> &gt;90'
)
CHAIN_COLORS = ["#4A90E2", "#F5A623", "#50E3C2", "#B886D8", "#7FB3E8", "#E8836D"]


def plddt_hex(v):
    """Convert pLDDT score to hex color."""
    if v >= 90:
        return "#0053D6"
    if v >= 70:
        return "#65CBF3"
    if v >= 50:
        return "#FFDB13"
    return "#FF7D45"


def _view(result, width, height):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(result.complex.to_mmcif(), "mmcif")
    return view


def show_plddt(result, width=700, height=480):
    """Cartoon coloured by per-residue confidence."""
    view = _view(result, width, height)
    view.setStyle({}, {})
    for i, score in enumerate(np.asarray(result.plddt).ravel() * 100):
        view.setStyle({"resi": i + 1}, {"cartoon": {"color": plddt_hex(score)}})
    view.addStyle({"hetflag": True}, {"stick": {}})
    view.zoomTo()
    display(HTML(PLDDT_LEGEND))
    return view


def show_chains(result, width=700, height=480):
    """One colour per chain; ligands and ions drawn as sticks."""
    view = _view(result, width, height)
    for i, chain in enumerate(sorted(result.complex.metadata.chain_lookup.values())):
        color = CHAIN_COLORS[i % len(CHAIN_COLORS)]
        view.setStyle({"chain": chain}, {"cartoon": {"color": color}})
    view.addStyle({"hetflag": True}, {"stick": {}})
    view.zoomTo()
    return view

## 1. The pattern

Every fold in this notebook is the same three steps:

1. **Say what you want** — build a `StructurePredictionInput` listing your molecules.
2. **Fold it** — `builder.fold(model, ...)`.
3. **Look at it** — coordinates and confidence come back together.

Here it is for ubiquitin, a small protein you may know.


In [ ]:
from esm.models.esmfold2 import ProteinInput, StructurePredictionInput

UBIQUITIN = (
    "MQIFVKTLTGKTITLEVEPSDTIENVKAKIQDKEGIPPDQQRLIFAGKQLEDGRTLSDYNIQKESTLHLVLRLRGG"
)

# 1. say what you want
spi = StructurePredictionInput(sequences=[ProteinInput(id="A", sequence=UBIQUITIN)])

# 2. fold it
result = builder.fold(model, spi, num_loops=10, num_sampling_steps=100, seed=0)

# 3. look at it
print(f"pLDDT {float(result.plddt.mean()):.3f}   pTM {float(result.ptm):.3f}")
show_plddt(result)

**The colours are the model telling you how much to trust each part.** Blue is
confident, red-orange is not. Here the body of the protein is blue and the tail
at the end runs orange — and it should, because that tail really is floppy in
solution. It is the part that gets attached to other proteins.

Two numbers come with it:

- **pLDDT** — confidence per residue, 0 to 1.
- **pTM** — one number for the shape overall.

That is the whole pattern. Everything below only changes **step 1**: what goes in
the list.


## 2. Add more molecules

Proteins rarely work alone. To fold something bound to RNA and DNA, put the RNA
and the DNA in the list too.

This is RNase H1 (PDB 4H8K), an enzyme that grabs an RNA/DNA hybrid and cuts the
RNA strand. It works as a pair of identical copies, which is what `id=["A", "B"]`
means: **one sequence, present twice**.


In [ ]:
from esm.models.esmfold2 import DNAInput, RNAInput

RNASEH = (
    "MNKIIIYTDGGARGNPGPAGIGVVITDEKGNTLHESSAYIGETTNNVAEYEALIRALEDLQ"
    "MFGDKLVDMEVEVRMNSELIVRQMQGVYKVKEPTLKEKFAKIAHIKMERVPNLVFVHIPRE"
    "KNARADELVNEAIDKALS"
)

spi = StructurePredictionInput(
    sequences=[
        ProteinInput(id=["A", "B"], sequence=RNASEH),  # the enzyme, two copies
        RNAInput(id="C", sequence="CGACACCUGAUUCC"),  # the strand it cuts
        DNAInput(id="D", sequence="GGAATCAGGTGTCG"),  # its partner strand
    ]
)

result = builder.fold(model, spi, num_loops=10, num_sampling_steps=100, seed=0)
print(f"pLDDT {float(result.plddt.mean()):.3f}   ipTM {float(result.iptm):.3f}")
show_chains(result)

Two protein chains, and the RNA/DNA duplex threaded through the middle.

A third number showed up: **ipTM**. Whenever there is more than one molecule, this
is the one to read — it says whether they are **positioned correctly against each
other**, not just folded correctly on their own.


## 3. Add a drug

Semaglutide, bound to its receptor. Three new things, one line each:

- The peptide has one **unnatural amino acid** at position 1. Non-standard
  building blocks are named by their PDB code — here `AIB`.
- It carries a **fatty-acid tail** that is not a protein at all. Anything you can
  draw as a SMILES string can go in the list.
- That tail is **chemically bonded** to the peptide, so we say which atoms join.


In [ ]:
from esm.models.esmfold2 import CovalentBond, LigandInput, Modification

RECEPTOR = (
    "MKTIIALSYIFCLVFADYKDDDDLEVLFQGPARPQGATVSLWETVQKWREYRRQCQRSLTEDPPPATDLFCNRTFDEYAC"
    "WPDGEPGSFVNVSCPWYLPWASSVPQGHVYRFCTAEGLWLQKDNSSLPWRDLSECEESKRGERSSPEEQLLFLYIIYTVG"
    "YALSFSALVIASAILLGFRHLHCTRNYIHLNLFASFILRALSVFIKDAALKWMYSTAAQQHQWDGLLSYQDSLSCRLVFL"
    "LMQYCVAANYYWLLVEGVYLYTLLAFSVFSEQWIFRLYVSIGWGVPLLFVVPWGIVKYLYEDEGCWTRNSNMNYWLIIRL"
    "PILFAIGVNFLIFVRVICIVVSKLKANLMCKTDIKCRLAKSTLTLIPLLGTHEVIFAFVMDEHARGTLRFIKLFTELSFT"
    "SFQGLMVAILYCFVNNEVQLEFRKSWERWRLEHLHIQRDSSMKPLKCPTSSLSSGATAGSSMYTATCQASCSPAGLEVLF"
    "QGPHHHHHHH"
)
PEPTIDE = "HAEGTFTSDVSSYLEGQAAKEFIAWLVRGRG"
FATTY_TAIL = (
    "C(=O)(CCOCCOCC(=O)NCCOCCOCCNCC(=O)N[C@@H](CCC(=O)NCCCCCCCCCCCCCCCCCC(=O)O)C(=O)O)"
)

lysine = PEPTIDE.index("K")  # the tail attaches here

spi = StructurePredictionInput(
    sequences=[
        ProteinInput(id="A", sequence=RECEPTOR),
        ProteinInput(
            id="B",
            sequence=PEPTIDE,
            modifications=[Modification(position=1, ccd="AIB")],
        ),
        LigandInput(id="C", smiles=FATTY_TAIL),
    ],
    covalent_bonds=[
        CovalentBond(
            chain_id1="B",
            res_idx1=lysine,
            atom_idx1=8,  # the lysine's N
            chain_id2="C",
            res_idx2=0,
            atom_idx2=0,
        )  # the tail's first atom
    ],
)

result = builder.fold(model, spi, num_loops=10, num_sampling_steps=100, seed=0)
print(f"pLDDT {float(result.plddt.mean()):.3f}   ipTM {float(result.iptm):.3f}")
show_chains(result)

The peptide sits in the receptor, with the fatty tail trailing off it. That tail
is why semaglutide is injected weekly instead of daily — it sticks to a blood
protein and slows the drug's clearance.

Confidence is lower here than before, and fairly so: this is a membrane receptor
folded without a membrane, and a long floppy tail that genuinely has no single
shape. **A lower number here is the model being honest, not being wrong.**


## 4. Add evolution

Everything so far used one sequence per molecule. You can also hand the model an
**alignment** — the same protein from hundreds of other species.

Why that helps: if two positions in a protein always mutate together across
evolution, they are probably touching. An alignment carries that signal, and for
two chains it also hints at how they meet.

**One thing to know:** the `-Fast` checkpoint has no alignment reader, and will
ignore an MSA without complaining. For alignments, load the full model.


In [ ]:
model = EsmFold2Model.from_pretrained("biohub/ESMFold2", device="cuda").eval()
model.set_kernel_backend("fused")

In [ ]:
BASE = "https://raw.githubusercontent.com/Biohub/esm/main/cookbook/tutorials"
!wget -q {BASE}/g3l5_chainA.a3m -O chainA.a3m
!wget -q {BASE}/g3l5_chainB.a3m -O chainB.a3m

In [ ]:
from esm.utils.msa import MSA

# A two-protein complex from vaccinia virus (PDB 7YTU), one alignment per chain.
CHAIN_A = (
    "GPYYPTNKLQAAVMETDRENAIIRQRNDEIPTRTLDTAIFTDASTVASAQIHLYYNSNIGKII"
    "MSLNGKKHTFNLYDDNDIRTLLPILLLSK"
)
CHAIN_B = (
    "GPNMFFMPKRKIPDPIDRLRRANLACEDDKLMIYGLPWMTTQTSALSINSKPIVYKDCAKLLRSINGSQPVSLNDVLRR"
)

spi = StructurePredictionInput(
    sequences=[
        ProteinInput(id="A", sequence=CHAIN_A, msa=MSA.from_a3m(path="chainA.a3m")),
        ProteinInput(id="B", sequence=CHAIN_B, msa=MSA.from_a3m(path="chainB.a3m")),
    ]
)

result = builder.fold(model, spi, num_loops=10, num_sampling_steps=100, seed=0)
print(f"pLDDT {float(result.plddt.mean()):.3f}   ipTM {float(result.iptm):.3f}")
show_chains(result)

An alignment is just a text file (`.a3m`) — one you already have if you have ever
run a database search. Loading it is one line, and attaching it is one argument.


## 5. What you just did

Four structures — a protein, a protein–RNA–DNA complex, a drug bound to its
receptor, and a two-chain complex with evolutionary information — on one GPU, in
about a minute of compute.

And the code never really changed:

```python
spi = StructurePredictionInput(sequences=[ ...your molecules... ])
result = builder.fold(model, spi)
```

Everything specific lived in that list. **Adding a molecule to a structure
prediction is adding an item to a list.**

Reading the answer, briefly:

| | means |
|---|---|
| **pLDDT** | per-residue confidence. Above ~0.8, trust it |
| **pTM** | is the overall shape right |
| **ipTM** | *for complexes:* are the pieces placed correctly against each other |

Low confidence is information. It usually means "this part is genuinely
flexible", and that is often the biologically interesting answer.


## 6. If you want to go further

Things we did not need today, in rough order of how often people reach for them:

- **Ask for several answers at once.** `num_diffusion_samples=N` returns N
  structures; rank them by confidence and keep the best.
- **Think harder.** `num_loops` controls how many refinement passes the model
  makes. More can help on difficult targets.
- **Nudge it toward a known shape.** `distogram_conditioning` biases a chain
  toward geometry you already have.
- **Take the wrapper apart.** `builder.prepare_input()` and `builder.decode()` are
  the two halves of `fold()`, useful if you want the raw tensors.
- **Design binders.** ESMFold2 can be inverted to generate new binding proteins —
  there is a full protocol in the [binder design notebook](https://github.com/Biohub/esm/blob/main/cookbook/tutorials/binder_design.ipynb).
- **No NVIDIA GPU?** The companion notebook runs these same folds in MLX on an
  Apple Silicon Mac.

**Links** — [model card](https://huggingface.co/biohub/ESMFold2) ·
[code](https://github.com/Biohub/esm) ·
[full tutorial](https://github.com/Biohub/esm/blob/main/cookbook/tutorials/esmfold2.ipynb) ·
[preprint](https://www.biorxiv.org/content/10.64898/2026.06.03.729735) ·
[Slack](https://bit.ly/esm-slack)
